# 1. Sequential workflow in Langgraph 

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict 

In [15]:
class States(TypedDict):
    math : float 
    science : float 
    english : float 
    sanskrit : float 
    SS : float 
    result : float

In [28]:
def calculate_percentage(state : States) -> States:
    maths_marks = state['math'] 
    sci_marks = state['science']
    eng_marks = state['english']
    san_marks = state['sanskrit']
    ss_marks = state['SS']
    result = (((maths_marks + sci_marks + eng_marks + san_marks + ss_marks) / 100 ) / 5 ) * 100
    state['result'] = round(result,2)
    return state 

In [29]:
graph = StateGraph(States) 

graph.add_node("calculate_percentage",calculate_percentage)
graph.add_edge(START,'calculate_percentage')
graph.add_edge('calculate_percentage',END)

workflow = graph.compile()

In [30]:
initial_state = {
    "math" : 98,
    "science" : 99,
    "english" : 97,
    "sanskrit" : 96,
    "SS" : 87,
}

final_state = workflow.invoke(initial_state)
print(final_state)

{'math': 98, 'science': 99, 'english': 97, 'sanskrit': 96, 'SS': 87, 'result': 95.4}


# S1mple LLM workflow

In [11]:
from langgraph.graph import StateGraph,START,END
from langchain_mistralai import ChatMistralAI 
from dotenv import load_dotenv
from typing import TypedDict  
load_dotenv()

True

In [4]:
class States(TypedDict):
    question : str 
    answer : str 


In [6]:
def ask_llm(state : States) -> States : 

    question = state['question']
    
    llm = ChatMistralAI(
        model = "mistral-small-latest"
    )

    prompt = f"answer the following question : {question}" 
    response = llm.invoke(prompt) 
    state['answer'] = response.content
    return state 

In [8]:
graph = StateGraph(States)

graph.add_node("ask_llm",ask_llm)
graph.add_edge(START,"ask_llm")
graph.add_edge("ask_llm",END)

workflow = graph.compile()

In [12]:
initial_state = {
    "question" : "tell me about Chander Bing"
}

response = workflow.invoke(initial_state)

In [13]:
print(f"Question : {response['question']}\nAnswer : {response['answer']}")

Question : tell me about Chander Bing
Answer : Chander Bing is a fictional character from the popular American sitcom *Friends*, which aired from 1994 to 2004. He is portrayed by actor Matthew Perry, who became widely known for his role as the witty, sarcastic, and often neurotic Chandler Muriel Bing.

### **Character Overview:**
- **Full Name:** Chandler Muriel Bing
- **Occupation:** Originally a statistical analysis and data reconfiguration specialist (later a copywriter and advertising executive)
- **Personality:** Known for his sarcasm, humor, and social awkwardness, Chandler often uses humor as a defense mechanism. Despite his insecurities, he is loyal, caring, and has a strong moral compass.
- **Relationships:**
  - Married to Monica Geller (played by Courteney Cox).
  - Close friends with Ross, Rachel, Joey, Phoebe, and Monica (the core *Friends* gang).
  - Struggled with commitment issues early in the series but eventually found stability with Monica.

### **Key Storylines & Tr

In [24]:
class States(TypedDict):
    title : str 
    outline : str
    content : str 
    quote : str  


In [25]:
def create_outline(state : States) -> States:

    title = state['title']
    
    llm = ChatMistralAI(
        model = "mistral-small-latest"
    )

    prompt = f"Generate a detail outline for a research paper on the topic : {title}"

    response = llm.invoke(prompt)

    state['outline'] = response.content 

    return state 


In [26]:
def create_content(state : States) -> States:
    title = state['title']
    outline = state['outline'] 

    llm = ChatMistralAI(
        model = "mistral-small-latest"
    )

    prompt = f"based on title : {title} and outline : {outline} generate a detail research content."

    response = llm.invoke(prompt) 

    state['content'] = response.content 

    return state 

In [30]:
def create_quote(state : States) -> States:
    title = state['title']
    outline = state['outline']

    llm = ChatMistralAI(
        model = 'mistral-small-latest'
    )

    prompt = f"based on title : {title} and outline : {outline} generate a create motivation quote."

    state['quote'] = llm.invoke(prompt).content

    return state

In [31]:
graph = StateGraph(States) 

graph.add_node("TitleCreator",create_outline)
graph.add_node("OutlineCreator",create_content)
graph.add_node("QuoteCreator",create_quote) 

graph.add_edge(START,"TitleCreator")
graph.add_edge("TitleCreator",'OutlineCreator')
graph.add_edge("OutlineCreator",'QuoteCreator')
graph.add_edge("QuoteCreator",END)

workflow = graph.compile()

In [32]:
initial_state = {
    "title" : "LIFE"
}

response = workflow.invoke(initial_state)

In [34]:
print(f"Title : {response['title']}")
print("="*100)
print(f"Outline :\n{response['outline']}")
print("="*100)
print(f"Content :\n{response['content']}")
print("="*100)
print(f"quote :\n{response['quote']}")


Title : LIFE
Outline :
# **Research Paper Outline: Life**

## **Title**
*"Exploring the Multidimensional Nature of Life: From Biological Foundations to Philosophical Implications"*

## **Abstract**
Brief summary of the paper’s objectives, key findings, and conclusions.

---

## **I. Introduction**
### **A. Background and Context**
1. Definition of "life" in various disciplines (biology, philosophy, theology, etc.)
2. Historical perspectives on the concept of life (ancient vs. modern views)
3. Importance of studying life (scientific, philosophical, ethical implications)

### **B. Research Objectives**
1. To analyze the biological, philosophical, and existential dimensions of life.
2. To explore the origins of life and theories of abiogenesis.
3. To examine the ethical and societal implications of life (e.g., artificial life, bioethics).
4. To discuss the future of life (space exploration, transhumanism, sustainability).

### **C. Thesis Statement**
*"Life is a complex, multifaceted phen

# 2. Parallel Workflow 

In [ ]:
from langgraph.graph import StateGraph,START,END 
from typing import TypedDict, Annotated
from pydantic import BaseModel,Field 
import operator

In [58]:
class BatsmenState(TypedDict) : 
    runs : int 
    sixes : int 
    fours : int 
    balls : int 

    ball_per_boundary : float 
    strike_rate : float
    boundary_rate : float 
    summary : str 
    

In [86]:
def calculate_ball_per_boundary(state: BatsmenState):
    balls = state['balls']
    four = state['fours']
    six = state['sixes'] 
    res = (four + six) / balls 
    return {"ball_per_boundary": res}

def calculate_strike_rate(state: BatsmenState):
    res = (state['runs'] / state['balls'])* 100
    return {"strike_rate": res}

def calculate_boundary_rate(state: BatsmenState):
    result = (((state['fours'] * 4 + state['sixes']*6)) / (state['runs'])) * 100
    return {"boundary_rate": result}

def Generate_summary(state: BatsmenState):
    summary = (f"The player made {state['runs']} in {state['balls']} balls.\n"
               f"Strike Rate: {state['strike_rate']}\n"
               f"Boundary rate: {state['boundary_rate']}\n"
               f"Ball per boundary: {state['ball_per_boundary']}")
    return {"summary": summary}

In [87]:
graph = StateGraph(BatsmenState)

graph.add_node("calculate_ball_per_boundary",calculate_ball_per_boundary)
graph.add_node("calculate_strike_rate",calculate_strike_rate)
graph.add_node("calculate_boundary_rate",calculate_boundary_rate)
graph.add_node("Generate_Summary",Generate_summary)

graph.add_edge(START,"calculate_ball_per_boundary")
graph.add_edge(START,"calculate_strike_rate")
graph.add_edge(START,"calculate_boundary_rate")

graph.add_edge("calculate_ball_per_boundary","Generate_Summary")
graph.add_edge("calculate_strike_rate","Generate_Summary")
graph.add_edge("calculate_boundary_rate","Generate_Summary")

graph.add_edge("Generate_Summary",END)

workflow = graph.compile()

In [88]:
initial_state = {
    "runs" : 264, 
    "sixes" : 9, 
    "fours" : 33,
    "balls" : 173 ,
}

response = workflow.invoke(initial_state)
print(response['summary'])

The player made 264 in 173 balls.
Strike Rate: 152.60115606936415
Boundary rate: 70.45454545454545
Ball per boundary: 0.24277456647398843


In [92]:
class formatted_output(BaseModel):
    feedback : str = Field(description="Detailed feedback of given essay")
    score : int = Field(description="Score out of 10",ge=0,lt=10)


In [93]:
model = ChatMistralAI(
    model = "mistral-small-latest"
)
structured_model = model.with_structured_output(formatted_output) 

In [97]:
class Essay_evalution(TypedDict):
    essay : str 
    language_feedback : str 
    analysis_feedback : str 
    clarity_feedback : str 
    overall_feedback : str 
    individual_score : Annotated[list[int],operator.add]
    average_score = float 

In [102]:
def evaluate_language_feedback(state : Essay_evalution) -> Essay_evalution : 

    prompt = f"Evaluate the language quality of the following essay and provide a detailed feedback and assign a score from 10. Essay : {state['essay']}"

    response = structured_model.invoke(prompt)

    return {"language_feedback" : response.feedback, "individual_score" : [response.score]} 

def evaluate_analysis_feedback(state : Essay_evalution) -> Essay_evalution:

    prompt = f"Evaluate the depth of analysis of the following essay and provide a detailed feedback and assign a score from 10. Essay : {state['essay']}"

    response = structured_model.invoke(prompt)

    return {"analysis_feedback" : response.feedback, "individual_score" : [response.score]}

def evaluate_clarity_feedback(state : Essay_evalution) -> Essay_evalution: 

    prompt = f"Evaluate the clarity of thought of the following essay and provide a detailed feedback and assign a score from 10. Essay : {state['essay']}"
    
    response = structured_model.invoke(prompt)

    return {"clarity_feedback" : response.feedback, "individual_score" : [response.score]}

def evaluate_final_feedback(state : Essay_evalution) -> Essay_evalution: 

    prompt = f"based on following feedbacks create a summarized feedback \n Language Feedback : {state['language_feedback']}, Analysis Feedback : {state['analysis_feedback']}, Clarity Feedback : {state['clarity_feedback']}."

    overall_feedback = structured_model.invoke(prompt).feedback 

    avg_score = sum(state['individual_score']) / len(state['individual_score']) 

    return {"overall_feedback": overall_feedback, "average_score" : avg_score}

In [103]:
graph = StateGraph(Essay_evalution) 

graph.add_node("evaluate_language_feedback",evaluate_language_feedback)
graph.add_node("evaluate_analysis_feedback",evaluate_analysis_feedback)
graph.add_node("evaluate_clarity_feedback",evaluate_clarity_feedback)
graph.add_node("evaluate_final_feedback",evaluate_final_feedback)

graph.add_edge(START,"evaluate_language_feedback")
graph.add_edge(START,"evaluate_analysis_feedback")
graph.add_edge(START,"evaluate_clarity_feedback")


graph.add_edge("evaluate_language_feedback","evaluate_final_feedback")
graph.add_edge("evaluate_analysis_feedback","evaluate_final_feedback")
graph.add_edge("evaluate_clarity_feedback","evaluate_final_feedback")

graph.add_edge("evaluate_final_feedback",END)

workflow = graph.compile()

In [104]:
initial_state = {
    "essay" : """Rohit Sharma: The Renaissance of Indian Cricket and the Art of Strategic Leadership
Introduction
In the socio-cultural fabric of India, cricket transcends the boundaries of mere sport to become a unifying national narrative. At the heart of this contemporary narrative stands Rohit Gurunath Sharma—a figure whose journey from the humble suburbs of Nagpur and Borivali to the pinnacle of global cricket epitomizes the "Indian Dream." Beyond the statistical milestones of double centuries and five IPL titles, Sharma represents a paradigm shift in leadership and the aesthetic evolution of the modern game. His career serves as a masterclass in resilience, adaptability, and the transition from individual brilliance to collective stewardship.

The Arc of Resilience: From "Talent" to "Tenacity"
For much of his early career, Rohit Sharma was burdened by the label of "prodigious talent"—a double-edged sword that often implies unfulfilled potential. His exclusion from the 2011 World Cup squad was a watershed moment in his personal history. In a classic example of "emotional intelligence," Sharma pivoted from despair to disciplined transformation. The decision to promote him to the opening slot in 2013 was not just a tactical masterstroke but a strategic realignment of his career. It teaches a vital lesson in public administration and life: that the right placement of human resources is essential for peak performance.

Leadership Philosophy: The "Cool" Commander
In the lexicon of leadership, if MS Dhoni was the "Stoic" and Virat Kohli the "Aggressor," Rohit Sharma is the "Collaborator." His captaincy is characterized by:

Empowerment of the Individual: Sharma is known for giving long ropes to young players, fostering a "psychological safety net" that allows them to fail before they flourish.

Data-Driven Intuition: He blends traditional "gut feel" with modern analytical rigor, particularly evident in his tenure with the Mumbai Indians and the National side.

Selfless Intent: His approach in the 2023 and 2024 ICC tournaments—sacrificing personal milestones for high-impact starts—redefined the "DNA" of Indian batting. This shift from "ego-centric" to "eco-centric" (team-first) performance is a hallmark of great statesman-like leadership.

Socio-Economic Impact and the Youth Icon
Rohit Sharma’s rise mirrors the democratization of Indian sports. As a product of a middle-class household, supported by his uncle’s meager savings, his success validates the "meritocratic" ideals of the Indian Constitution. Furthermore, his role as a WWF-India Rhino Ambassador and his advocacy for animal welfare highlight the modern athlete's role as a socially responsible citizen. He utilizes his "soft power" to influence environmental consciousness, bridging the gap between sports and sustainable development goals.

The Legacy of the "Hitman"
Statistically, Sharma’s records—three ODI double centuries, the highest individual score of 264, and leading India to the 2024 T20 World Cup and 2025 Champions Trophy—place him in the pantheon of greats. However, his true legacy lies in his "effortless elegance." In an era of high-velocity "power-hitting," his ability to find time against the fastest bowlers is a reminder that even in a fast-paced world, there is no substitute for grace and timing.

Conclusion
Rohit Sharma's journey is a testament to the fact that greatness is not a destination but a continuous process of evolution. He transitioned from a struggling middle-order batter to the "Hitman," and finally to the "Elder Statesman" of Indian cricket. As India strives to become a global sporting powerhouse, the "Rohit Model" of leadership—calm under fire, inclusive in decision-making, and selfless in execution—provides a blueprint for excellence across all spheres of national life. He remains, quite literally, the "Captain of Hearts" and a beacon of inspiration for the billion-plus dreams he carries."""
    }

In [105]:
response = workflow.invoke(initial_state)


In [106]:
response 

{'essay': 'Rohit Sharma: The Renaissance of Indian Cricket and the Art of Strategic Leadership\nIntroduction\nIn the socio-cultural fabric of India, cricket transcends the boundaries of mere sport to become a unifying national narrative. At the heart of this contemporary narrative stands Rohit Gurunath Sharma—a figure whose journey from the humble suburbs of Nagpur and Borivali to the pinnacle of global cricket epitomizes the "Indian Dream." Beyond the statistical milestones of double centuries and five IPL titles, Sharma represents a paradigm shift in leadership and the aesthetic evolution of the modern game. His career serves as a masterclass in resilience, adaptability, and the transition from individual brilliance to collective stewardship.\n\nThe Arc of Resilience: From "Talent" to "Tenacity"\nFor much of his early career, Rohit Sharma was burdened by the label of "prodigious talent"—a double-edged sword that often implies unfulfilled potential. His exclusion from the 2011 World C